# 00 — Проверка окружения и связи с Unity

**Гейт Фазы 1.** Ноутбук подтверждает, что установлен рабочий стек и что Python видит среду Unity — и в редакторе, и в headless-сборке.

Ноутбук выполняется сверху вниз, без ручных правок. Ячейка подключения к редактору потребует нажать **Play** в Unity — это единственное действие руками, и оно ожидаемо: Editor-режим для того и существует.

**Что проверяется**

| # | Ячейка | Что подтверждает |
|---|---|---|
| 1 | Версии | стек соответствует `docs/01_STACK.md` |
| 2 | Сиды и устройство | воспроизводимость настроена, GPU виден |
| 3 | Реестр сред | коды `E##` и пути билдов на месте |
| 4 | Контракт ONNX | экспорт и верификация работают **без** Unity |
| 5 | Build-режим | Python подключается к headless-сборке |
| 6 | Editor-режим | Python подключается к редактору |
| 7 | Итог | сводка: что прошло, что нет |

In [ ]:
# --- 0. Пути. Ноутбук не зависит от текущего каталога (требование 9.5) ---
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория: запустите ноутбук изнутри unity-ml-agents-lab")
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "python"))
print("корень репозитория:", REPO_ROOT)

checks: dict[str, bool] = {}

## 1. Версии стека

Версии не берутся из памяти — печатаются из установленных пакетов (требование 4.1).

In [ ]:
import platform

import numpy as np
import torch

print(f"python        {platform.python_version()}  ({sys.executable})")
print(f"numpy         {np.__version__}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}  доступна: {torch.cuda.is_available()}")

try:
    import mlagents_envs
    print(f"mlagents_envs {mlagents_envs.__version__}  (тег {mlagents_envs.__release_tag__})")
    checks["mlagents_envs установлен"] = True
except ImportError as exc:
    print("mlagents_envs НЕ установлен:", exc)
    checks["mlagents_envs установлен"] = False

import onnx, onnxruntime
print(f"onnx          {onnx.__version__}")
print(f"onnxruntime   {onnxruntime.__version__}")

if torch.cuda.is_available():
    print(f"GPU           {torch.cuda.get_device_name(0)}")

checks["python 3.10.x"] = platform.python_version().startswith("3.10.")

## 2. Сиды и устройство

In [ ]:
from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
device = resolve_device("auto")
print("устройство:", device)

# Один и тот же сид обязан давать одну и ту же выборку.
set_global_seed(SEED); a = torch.randn(3)
set_global_seed(SEED); b = torch.randn(3)
checks["сиды воспроизводимы"] = bool(torch.equal(a, b))
print("воспроизводимость сидов:", checks["сиды воспроизводимы"])

## 3. Реестр сред

In [ ]:
from labrl.envs.registry import REGISTRY

for entry in REGISTRY.values():
    has_folder = entry.unity_folder.is_dir()
    has_build = entry.build_path.is_file()
    print(f"{entry.env_id:22} {entry.status:12} папка: {'да ' if has_folder else 'нет'}  билд: {'да' if has_build else 'нет'}")

checks["реестр сред читается"] = len(REGISTRY) > 0

## 4. Контракт ONNX без Unity

Самая ценная проверка этого ноутбука: она ловит поломку экспорта **до** того, как на обучение потрачены часы. Unity для неё не нужен.

In [ ]:
import tempfile

import torch.nn as nn

from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

policy = nn.Sequential(nn.Linear(25, 32), nn.ReLU(), nn.Linear(32, 4))
spec = ActionSpecLite(discrete_branches=(4,))
obs_shapes = [(25,)]

with tempfile.TemporaryDirectory() as tmp:
    path = export_policy_to_onnx(policy, spec, obs_shapes, Path(tmp) / "probe.onnx")
    sample = [np.random.default_rng(0).standard_normal((64, 25)).astype(np.float32)]
    result = verify_onnx_model(path, policy, spec, obs_shapes, sample_obs=sample)

print(result.report())
checks["контракт ONNX соблюдён"] = result.passed

## 5. Build-режим

Подключение к headless-сборке. Если билда нет, ячейка честно сообщает об этом и не притворяется успешной.

Собрать среду: `python scripts/build_env.py E01_GridWorld`

In [ ]:
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E01_GridWorld"
build_path = get_env(ENV_ID).build_path

if not build_path.is_file():
    print(f"билд не найден: {build_path}")
    print(f"соберите его: python scripts/build_env.py {ENV_ID}")
    checks["Build-режим"] = False
else:
    with open_unity_env(ENV_ID, build_path=build_path, seed=SEED, time_scale=20.0) as handle:
        print(handle.describe())
        vec = VecUnityEnv(handle)
        obs = vec.reset()
        print(f"\nарен (параллельных сред): {vec.num_envs}")
        print(f"формы наблюдений батчем: {[o.shape for o in obs]}")

        actions = np.zeros((vec.num_envs, len(vec.action_spec.discrete_branches)), dtype=np.int32)
        step = vec.step(actions)
        print(f"шаг выполнен: награда {step.reward}, активны {step.active}")
    checks["Build-режим"] = True

## 6. Editor-режим

После запуска ячейки переключитесь в Unity и нажмите **Play**. Ожидание — до 5 минут.

Поставьте `RUN_EDITOR_CHECK = False`, чтобы пропустить проверку (например, при прогоне в CI).

In [ ]:
RUN_EDITOR_CHECK = True

if not RUN_EDITOR_CHECK:
    print("пропущено по флагу RUN_EDITOR_CHECK")
    checks["Editor-режим"] = None
else:
    print("Откройте сцену среды в Unity и нажмите Play…")
    with open_unity_env(ENV_ID, build_path=None, seed=SEED) as handle:
        print(handle.describe())
        vec = VecUnityEnv(handle)
        obs = vec.reset()
        print(f"\nарен: {vec.num_envs}, формы наблюдений: {[o.shape for o in obs]}")
    checks["Editor-режим"] = True

## 7. Итог

In [ ]:
print("Проверки:\n")
for name, ok in checks.items():
    mark = "пропущено" if ok is None else ("OK" if ok else "ПРОВАЛ")
    print(f"  {mark:10} {name}")

failed = [name for name, ok in checks.items() if ok is False]
print()
if failed:
    print("Гейт Фазы 1 НЕ пройден. Не прошло:", ", ".join(failed))
else:
    print("Гейт Фазы 1 пройден: окружение работает, связь с Unity есть.")